In [2]:
"""
Logistic Regression from Scratch (NumPy)
=========================================
This module implements a Binary Logistic Regression classifier from first principles.

Mathematical Overview:
----------------------
1. Forward Pass (Hypothesis Function):
   z = X @ w + b
   ŷ = σ(z) = 1 / (1 + exp(-z))

2. Loss Function (Binary Cross-Entropy / Log Loss):
   J(w, b) = - (1/m) * ∑ [ y * log(ŷ) + (1 - y) * log(1 - ŷ) ]

3. Optimization (Gradient Descent):
   ∂J/∂w = (1/m) * X^T @ (ŷ - y)
   ∂J/∂b = (1/m) * ∑ (ŷ - y)

   w_new = w - α * (∂J/∂w)
   b_new = b - α * (∂J/∂b)

"""

import numpy as np


class LogisticRegressionScratch:
    """
    Binary Logistic Regression classifier built using NumPy.

    Parameters:
    -----------
    learning_rate : float, default=0.01
        The step size (α) used for gradient descent parameter updates.
    n_iters : int, default=1000
        Number of iterations/epochs to run gradient descent.

    Attributes:
    -----------
    weights : np.ndarray of shape (n_features,)
        Learned weights for input features.
    bias : float
        Learned bias (intercept) term.
    loss_history : list of float
        Stores binary cross-entropy loss value at each iteration for diagnostics.
    """

    def __init__(self, learning_rate: float = 0.01, n_iters: int = 1000):
        self.lr = learning_rate
        self.n_iters = n_iters
        self.weights = None
        self.bias = None
        self.loss_history = []

    def _sigmoid(self, z: np.ndarray) -> np.ndarray:
        """
        Computes the Sigmoid (logistic) activation function.

        Derivation / Property:
        ----------------------
        σ(z) = 1 / (1 + e^(-z))

        Maps any real-valued scalar/matrix to the range (0, 1), representing probability.
        We clip 'z' to prevent numerical overflow in np.exp(-z).
        """
        z_clipped = np.clip(z, -500, 500)
        return 1.0 / (1.0 + np.exp(-z_clipped))

    def _compute_loss(self, y_true: np.ndarray, y_pred: np.ndarray) -> float:
        """
        Computes the Binary Cross-Entropy Loss (Log Loss).

        Mathematical Formula:
        ---------------------
        J(w, b) = - (1/m) * ∑ [ y * log(ŷ) + (1 - y) * log(1 - ŷ) ]

        Note: Epsilon (1e-15) is added to avoid log(0) undefined errors.
        """
        m = y_true.shape[0]
        epsilon = 1e-15
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
        loss = - (1 / m) * np.sum(
            y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred)
        )
        return loss

    def fit(self, X: np.ndarray, y: np.ndarray) -> "LogisticRegressionScratch":
        """
        Fit the model according to the given training data using Gradient Descent.

        Derivation of Gradients:
        ------------------------
        1. Chain Rule for Loss w.r.t weights (w_j):
           ∂J/∂w_j = (∂J/∂ŷ) * (∂ŷ/∂z) * (∂z/∂w_j)

        2. Step-by-Step Partial Derivatives:
           a) ∂J/∂ŷ = - (y / ŷ) + ((1 - y) / (1 - ŷ))
                    = (ŷ - y) / [ ŷ * (1 - ŷ) ]

           b) ∂ŷ/∂z = ∂/∂z [ 1 / (1 + e^-z) ]
                    = e^-z / (1 + e^-z)^2
                    = (1 / (1 + e^-z)) * (e^-z / (1 + e^-z))
                    = ŷ * (1 - ŷ)

           c) ∂z/∂w_j = ∂/∂w_j [ w_1*x_1 + ... + w_j*x_j + b ]
                      = x_j

        3. Multiplying together (Chain Rule):
           ∂J/∂w_j = [ (ŷ - y) / (ŷ * (1 - ŷ)) ] * [ ŷ * (1 - ŷ) ] * x_j
                   = (ŷ - y) * x_j

        4. Vectorized Gradient over all 'm' samples:
           ∂J/∂w = (1 / m) * X^T @ (ŷ - y)
           ∂J/∂b = (1 / m) * ∑ (ŷ - y)
        """
        n_samples, n_features = X.shape

        # Initialize parameters: zero initialization is safe for Logistic Regression
        self.weights = np.zeros(n_features)
        self.bias = 0.0
        self.loss_history = []

        # Optimization loop using Batch Gradient Descent
        for _ in range(self.n_iters):
            # Step 1: Linear combination (z = Xw + b)
            linear_model = np.dot(X, self.weights) + self.bias

            # Step 2: Pass through Sigmoid activation to get probabilities (ŷ)
            y_predicted = self._sigmoid(linear_model)

            # Step 3: Compute gradients via vectorization
            dw = (1 / n_samples) * np.dot(X.T, (y_predicted - y))
            db = (1 / n_samples) * np.sum(y_predicted - y)

            # Step 4: Update weights and bias (Gradient Descent Rule)
            self.weights -= self.lr * dw
            self.bias -= self.lr * db

            # Step 5: Track loss for evaluation/plotting
            current_loss = self._compute_loss(y, y_predicted)
            self.loss_history.append(current_loss)

        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """
        Predict probability estimates for input samples.

        Returns:
        --------
        np.ndarray of shape (n_samples,): Probabilities for class 1.
        """
        linear_model = np.dot(X, self.weights) + self.bias
        return self._sigmoid(linear_model)

    def predict(self, X: np.ndarray, threshold: float = 0.5) -> np.ndarray:
        """
        Predict binary class labels for input samples.

        Parameters:
        -----------
        X : np.ndarray of shape (n_samples, n_features)
        threshold : float, default=0.5
            Probability threshold to decide positive class (1).

        Returns:
        --------
        np.ndarray of shape (n_samples,): Predicted binary classes (0 or 1).
        """
        y_predicted_cls = self.predict_proba(X)
        return np.where(y_predicted_cls >= threshold, 1, 0)


# =====================================================================
# Example Usage / Test Driver (Ideal for README.md demonstrate section)
# =====================================================================
if __name__ == "__main__":
    from sklearn.datasets import make_classification
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score, classification_report

    # 1. Generate Synthetic Binary Classification Dataset
    X, y = make_classification(
        n_samples=1000,
        n_features=5,
        n_classes=2,
        random_state=42
    )

    # 2. Train / Test Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # 3. Instantiate & Train Model
    clf = LogisticRegressionScratch(learning_rate=0.1, n_iters=1000)
    clf.fit(X_train, y_train)

    # 4. Make Predictions
    predictions = clf.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)

    # 5. Display Results
    print(f"Model Training Complete!")
    print(f"Final Loss: {clf.loss_history[-1]:.4f}")
    print(f"Test Accuracy: {accuracy * 100:.2f}%\n")
    print("Classification Report:")
    print(classification_report(y_test, predictions))

Model Training Complete!
Final Loss: 0.3514
Test Accuracy: 88.00%

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.92      0.88        97
           1       0.92      0.84      0.88       103

    accuracy                           0.88       200
   macro avg       0.88      0.88      0.88       200
weighted avg       0.88      0.88      0.88       200

